## Hướng dẫn làm bài:
1. Sử dụng Pipeline để train mô hình
2. Lưu mô hình sau khi train và các thông tin mô tả vào file model.pkl
3. Nộp file .ipynb lên LMS, không cần nén
4. Copy file model.pkl vào thư mục T:\trained\

In [4]:
# Khai báo thông tin sinh viên
Lop = "1234"
Nhom = "123"
MSSV = "123456"
HoTen = "Nguyen Van A"
SoMay = 0

In [5]:
# ==========================
# 1. Import libraries
# ==========================
import pandas as pd
import joblib
import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# ==========================
# 2. Load dataset
# ==========================
train_file = "BMW_Car_Sales_Classification.csv"

train_df = pd.read_csv(train_file)

# Assume last column is the target
target_col = train_df.columns[-1]

# Split train dataset into 80% training, 20% validation
X = train_df.drop(columns=[target_col])
y = train_df[target_col]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==========================
# 3. Preprocessing
# ==========================
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# ==========================
# 4. Build pipeline with model
# ==========================
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# ==========================
# 5. Train and validate
# ==========================
clf.fit(X_train, y_train)

# Validation evaluation
y_val_pred = clf.predict(X_val)
print("Validation set performance:")
print(classification_report(y_val, y_val_pred))

# ==========================
# 6. Save model with metadata
# ==========================
metadata = {
    "Lop": Lop,
    "Nhom": Nhom,
    "MSSV": MSSV,
    "HoTen": HoTen,
    "SoMay": SoMay,
    "TaoLuc": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

save_obj = {"model": clf, "metadata": metadata}
joblib.dump(save_obj, "model.pkl")

print("Model and metadata saved to model.pkl")

Validation set performance:
              precision    recall  f1-score   support

        High       1.00      1.00      1.00      3049
         Low       1.00      1.00      1.00      6951

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000

Model and metadata saved to model.pkl
